# Agentic Chatbot Workflow with LangGraph

This notebook walks through building a stateful, multi-turn AI chatbot workflow using:
- **LangGraph** (`StateGraph`, `START`, `END`)
- **Google Gemini** (`ChatGoogleGenerativeAI`)
- **Memory Checkpointing** (`MemorySaver`)


In [ ]:
# 1. Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
print('GEMINI_API_KEY loaded:', bool(os.environ.get('GEMINI_API_KEY')))


In [ ]:
# 2. Initialize Google Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    api_key=gemini_key,
    temperature=0.7
)

# Quick test
test_res = llm.invoke('Say hello in one short sentence!')
print(test_res.content)


In [ ]:
# 3. Define State and Chat Node
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    # add_messages appends incoming messages rather than replacing
    messages: Annotated[List[BaseMessage], add_messages]

def chat_node(state: ChatState):
    response = llm.invoke(state['messages'])
    return {'messages': [response]}


In [ ]:
# 4. Construct and Compile StateGraph with Checkpointer
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(ChatState)
workflow.add_node('chat_node', chat_node)
workflow.add_edge(START, 'chat_node')
workflow.add_edge('chat_node', END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)
print('LangGraph chatbot compiled successfully!')


In [ ]:
# 5. Multi-Turn Conversation Memory Test
config = {'configurable': {'thread_id': 'demo-session-1'}}

# Turn 1
output1 = app.invoke(
    {'messages': [HumanMessage(content='Hello! My name is Taslim.')]},
    config=config
)
print('Turn 1 AI:', output1['messages'][-1].content)
print('-' * 50)

# Turn 2: Verify memory retention across turns
output2 = app.invoke(
    {'messages': [HumanMessage(content='Do you remember my name?')]},
    config=config
)
print('Turn 2 AI:', output2['messages'][-1].content)


In [ ]:
# 6. Streaming Responses
config = {'configurable': {'thread_id': 'demo-session-1'}}

print('Streaming reply:')
for event in app.stream(
    {'messages': [HumanMessage(content='Give me 3 quick tips for learning AI agents.')]},
    config=config,
    stream_mode='messages'
):
    msg, metadata = event
    if isinstance(msg, AIMessage) and msg.content:
        print(msg.content, end='', flush=True)
print()
